In [8]:
%load_ext autoreload
%autoreload 2
import numpy as np
from sys import path
path.insert(0, '..')
from scripts.logistic_reg import logistic_regression_loss,get_L_one_hot,train_log_reg
from scripts.utility import vCol,map_labels_into_numbers
import scipy
import json
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
D=np.load('../data/processed/D_train_latent.npy')
L=np.load('../data/processed/L_train.npy')
labels_to_num=json.load(open('../data/processed/labels_to_num.json','r'))
num_to_labels=json.load(open('../data/processed/num_to_label.json','r'))
D_test=np.load('../data/processed/D_test.npy')
L_test=np.load('../data/processed/L_test.npy')

In [7]:
D

array([[ 4.90444361,  3.62614966,  3.72809099, ..., -7.01448752,
        -6.84106682, -6.48973027],
       [-4.63251198, -5.74676351, -5.74217453, ...,  1.09870867,
         0.99162572,  0.75265794],
       [-0.92876343, -0.1555162 , -0.43350918, ...,  0.45251673,
         0.52603201,  0.40703671],
       [ 0.16578367,  0.19910355,  0.12028995, ...,  2.08591049,
         2.45074201,  2.88660849],
       [ 0.96125541, -1.1642225 ,  0.28296175, ..., -0.34692774,
        -0.59530938, -0.55492143]], shape=(5, 7352))

In [20]:
lambdas=[1e-4, 1e-2, 0.1, 1.0]
best_w=0
best_b=0
best_acc=0
for l in lambdas:
    w,b=train_log_reg(D,L,l)
    s=w.T @ D_test + b
    predictions=np.argmax(s,axis=0)
    accuracy=np.mean(predictions==L_test)
    if accuracy > best_acc:
        best_acc=accuracy
        best_w=w
        best_b=b
    print(f"Accuracy with lambda= {l} is: {accuracy*100}%")

Accuracy with lambda= 0.0001 is: 86.01968103155751%
Accuracy with lambda= 0.01 is: 87.20732948761453%
Accuracy with lambda= 0.1 is: 87.10553104852391%
Accuracy with lambda= 1.0 is: 84.8320325755005%


In [22]:
import numpy as np

def export_weights_to_rust(W, b, filename="model_weights.rs"):
    with open(filename, "w") as f:
        f.write("// File autogenerato: Parametri della Regressione Logistica\n")
        f.write("#![allow(dead_code)]\n\n")
        
        # Esporta la matrice dei pesi W (5x6) appiattita in un array 1D
        w_flat = W.flatten(order='C')
        f.write(f"pub static W_MATRIX: [f32; {len(w_flat)}] = [\n    ")
        w_str = ", ".join([f"{val:.6f}" for val in w_flat])
        f.write(w_str)
        f.write("\n];\n\n")
        
        # Esporta il vettore dei bias b (6x1)
        b_flat = b.flatten()
        f.write(f"pub static B_VECTOR: [f32; {len(b_flat)}] = [\n    ")
        b_str = ", ".join([f"{val:.6f}" for val in b_flat])
        f.write(b_str)
        f.write("\n];\n")
        
    print(f"Pesi esportati con successo in {filename}")

# Richiamo della funzione passando i parametri ottimali calcolati
export_weights_to_rust(best_w, best_b)

Pesi esportati con successo in model_weights.rs
